[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gouravkhanijoe13/agentic-ai-lab/blob/main/Lesson_35_Parallel_Fan_out_and_Map_Reduce.ipynb)

# Lesson 35 — Parallel Fan-out & Map-Reduce across A2A Peers
**Track 2 · Multi-Agent Coordination — Lesson 4 of 5**

So far in Track 2 every coordination pattern has been *sequential*:

| Lesson | Topology | Coordination |
|---|---|---|
| L32 — A2A protocol | 1 ↔ 1 | one agent sends, one agent answers |
| L33 — Blackboard | N around a shared doc | controller picks **one** KS per tick, round-robin |
| L34 — Debate | 2 advocates + Judge | strictly alternating turns |

That covers *coordination*, but it leaves money — and wall-clock time — on the table. Many agent workloads are **embarrassingly parallel**:

- Summarize 20 documents
- Fact-check 15 claims
- Run the same reasoning prompt 5 times and take the majority vote (self-consistency)
- Fan a question out to 3 specialist agents and pick the best answer

If you call these one at a time you pay `N × per-call latency`. Fanned out concurrently you pay `max(per-call latency)` — same cost, ~N× faster.

This lesson teaches the **fan-out / map-reduce** shape:

1. `asyncio.gather` primer — the engine under every fan-out
2. `parallel_fan_out(items, worker, max_concurrent)` — reusable in-process helper
3. `map_reduce(items, mapper, reducer, on_failure)` — with partial-failure policies
4. `self_consistency_vote(prompt, n_samples)` — the celebrity special case
5. Wire it into the L32 A2A protocol — fan one task out across N remote peers
6. When fan-out *is* the wrong call (the blackboard vs map-reduce decision)
7. The seven failure modes that bite real systems


## 0 · Setup

We need: `anthropic`, `httpx`, `fastapi`, `uvicorn`, `nest_asyncio`, `pydantic`.

Set `ANTHROPIC_API_KEY` in **Colab → 🔑 Secrets** (toggle "Notebook access" on).

In [ ]:
!pip install anthropic httpx fastapi uvicorn nest_asyncio pydantic -q


In [ ]:
import os, sys, time, asyncio, json, uuid, statistics
from collections import Counter
from typing import Callable, Any, Awaitable

# API key — works in Colab via Secrets, falls back to env var locally
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    pass

assert os.environ.get("ANTHROPIC_API_KEY"), "Set ANTHROPIC_API_KEY (Colab Secrets or env)"

import nest_asyncio; nest_asyncio.apply()  # let asyncio.run() work inside Jupyter
from anthropic import Anthropic, AsyncAnthropic

HAIKU = "claude-haiku-4-5"
SONNET = "claude-sonnet-4-5"

client = Anthropic()
aclient = AsyncAnthropic()  # async client — required for true concurrency
print("ready.")


## 1 · `asyncio.gather` — the engine

Python is single-threaded, but `asyncio` lets a single thread juggle many *I/O-bound* tasks (like HTTP calls to Claude) concurrently. The trick is to **await** a list of coroutines through `asyncio.gather` — the event loop fires them all off and only blocks for the *slowest* one to return.

Two rules to internalize:

1. **Use the async client.** `Anthropic()` is sync and blocks the event loop. `AsyncAnthropic()` returns awaitables you can fan out.
2. **`return_exceptions=True`.** Without this flag, the *first* coroutine that raises cancels every other in-flight task. With it, exceptions come back inline in the result list and you decide what to do.

Let's prove the speed-up.

In [ ]:
SUMMARIZE_DOCS = [
    "The 2010s deep learning revolution began with AlexNet winning ImageNet in 2012, beating handcrafted features by a wide margin and demonstrating GPUs as the right substrate for neural network training.",
    "Word2vec (2013) and GloVe (2014) showed that dense vector representations of words capture analogical structure, planting the seeds of modern embedding-based retrieval.",
    "Sequence-to-sequence with attention (Bahdanau 2014) and the Transformer (Vaswani 2017) replaced recurrence with parallelizable attention and became the foundation of every modern LLM.",
    "BERT (2018) showed bidirectional masked-language pretraining beats left-to-right LMs on understanding tasks; GPT (2018, 2019, 2020) showed autoregressive pretraining scales remarkably with parameters and data.",
    "RLHF (Christiano 2017, InstructGPT 2022) closed the alignment gap between a raw next-token predictor and a helpful assistant, enabling the consumer-facing chatbot wave.",
    "Diffusion models (DDPM 2020) and latent diffusion (Stable Diffusion 2022) turned image generation from a research curiosity into a mainstream creative tool.",
]

async def summarize_one(doc: str) -> str:
    resp = await aclient.messages.create(
        model=HAIKU,
        max_tokens=80,
        messages=[{"role": "user", "content": f"Summarize in ONE sentence:\n\n{doc}"}],
    )
    return resp.content[0].text.strip()

# Sequential baseline
async def sequential():
    out = []
    for d in SUMMARIZE_DOCS:
        out.append(await summarize_one(d))
    return out

# Parallel via gather
async def parallel():
    return await asyncio.gather(*(summarize_one(d) for d in SUMMARIZE_DOCS))

t0 = time.time(); seq = asyncio.run(sequential());  seq_dt = time.time() - t0
t0 = time.time(); par = asyncio.run(parallel());    par_dt = time.time() - t0

print(f"Sequential: {seq_dt:.1f}s  for {len(SUMMARIZE_DOCS)} docs")
print(f"Parallel  : {par_dt:.1f}s  for {len(SUMMARIZE_DOCS)} docs")
print(f"Speed-up  : {seq_dt/par_dt:.1f}×\n")
for i, s in enumerate(par):
    print(f"[{i}] {s[:100]}")

# 💡 EXPERIMENT — bump SUMMARIZE_DOCS to 20 entries. The speed-up grows with N
# until you hit Anthropic's per-key rate limit; then it flattens. That's why we
# need a Semaphore (next cell).


## 2 · `parallel_fan_out` — reusable helper with back-pressure

Bare `asyncio.gather` has two flaws for production agent work:

1. **No concurrency cap.** Fanning 200 tasks at once will trip your rate limit and the *whole batch* errors. You want a sliding window of, say, 8 in-flight.
2. **No identity.** `gather` returns results in *input* order, but if you build the inputs lazily (e.g. from a generator) you lose the mapping. Capture stable ids explicitly.

The helper below fixes both — `asyncio.Semaphore` for the in-flight cap, an explicit `id` per task, and `return_exceptions=True` so one bad task doesn't poison the batch.

In [ ]:
from dataclasses import dataclass
from typing import TypeVar, Generic

T = TypeVar("T")
R = TypeVar("R")

@dataclass(frozen=True)
class FanoutResult(Generic[R]):
    id: str            # stable identifier supplied by caller
    ok: bool           # False if worker raised
    value: R | None    # worker output (None if !ok)
    error: str | None  # str(exception) if !ok
    latency_s: float

async def parallel_fan_out(
    items: list[tuple[str, T]],          # (id, payload) — id is yours to define
    worker: Callable[[T], Awaitable[R]],  # async fn payload → result
    *,
    max_concurrent: int = 8,
) -> list[FanoutResult[R]]:
    sem = asyncio.Semaphore(max_concurrent)

    async def run_one(item_id: str, payload: T) -> FanoutResult[R]:
        async with sem:
            t0 = time.time()
            try:
                value = await worker(payload)
                return FanoutResult(item_id, True, value, None, time.time() - t0)
            except Exception as e:
                return FanoutResult(item_id, False, None, f"{type(e).__name__}: {e}", time.time() - t0)

    return await asyncio.gather(*(run_one(i, p) for i, p in items))

# Smoke test — fan out summaries, return in same shape
async def demo():
    items = [(f"doc-{i}", d) for i, d in enumerate(SUMMARIZE_DOCS)]
    results = await parallel_fan_out(items, summarize_one, max_concurrent=4)
    for r in results:
        marker = "✓" if r.ok else "✗"
        snippet = (r.value or r.error or "")[:70]
        print(f"{marker} {r.id} ({r.latency_s:.1f}s)  {snippet}")

asyncio.run(demo())


## 3 · The map-reduce shape

`parallel_fan_out` is the **map** step — `N` workers run independently. A real workload usually wants a **reduce** step that consumes all the map outputs and produces a single answer (a summary of summaries, a vote tally, a fused report).

`map_reduce(items, mapper, reducer)` makes this explicit. The reducer is *another* agent — a different model, a different prompt, a different specialty.

Two important policy choices the reducer must take:

- **What does "enough" mean?** `all_or_nothing` says any failure poisons the run; `best_effort_min_k` requires at least k succeed; `vote_with_quorum` only reduces if a quorum agrees. Pick by domain — fact-checking wants high k, brainstorming is fine with best-effort.
- **Where does provenance live?** The reducer must receive the *stable id* of each map output so its summary can cite "Doc-3 said X, Doc-7 said Y". Hiding identity behind anonymous list positions makes audit impossible.

In [ ]:
from enum import Enum

class FailurePolicy(str, Enum):
    ALL_OR_NOTHING = "all_or_nothing"
    BEST_EFFORT = "best_effort"           # take whatever survived
    MIN_K = "min_k"                       # require at least k successful

@dataclass(frozen=True)
class MapReduceResult(Generic[R]):
    final: R
    n_total: int
    n_ok: int
    failed_ids: list[str]
    map_outputs: list[FanoutResult]       # full per-item record for audit
    map_latency_s: float
    reduce_latency_s: float

async def map_reduce(
    items: list[tuple[str, T]],
    mapper: Callable[[T], Awaitable[R]],
    reducer: Callable[[list[tuple[str, R]]], Awaitable[Any]],
    *,
    max_concurrent: int = 8,
    on_failure: FailurePolicy = FailurePolicy.BEST_EFFORT,
    min_k: int = 1,
) -> MapReduceResult:
    t0 = time.time()
    mapped = await parallel_fan_out(items, mapper, max_concurrent=max_concurrent)
    map_dt = time.time() - t0

    ok = [r for r in mapped if r.ok]
    failed = [r.id for r in mapped if not r.ok]

    if on_failure is FailurePolicy.ALL_OR_NOTHING and failed:
        raise RuntimeError(f"all_or_nothing breached, failed: {failed}")
    if on_failure is FailurePolicy.MIN_K and len(ok) < min_k:
        raise RuntimeError(f"min_k={min_k} not met, got {len(ok)} ok")

    t0 = time.time()
    final = await reducer([(r.id, r.value) for r in ok])
    reduce_dt = time.time() - t0

    return MapReduceResult(
        final=final,
        n_total=len(items),
        n_ok=len(ok),
        failed_ids=failed,
        map_outputs=mapped,
        map_latency_s=map_dt,
        reduce_latency_s=reduce_dt,
    )

# Concrete example — summarize 6 docs in parallel, then ask Sonnet (sharper
# synthesizer) to fuse them into a single themed paragraph, citing doc ids.
async def fuse_reducer(parts: list[tuple[str, str]]) -> str:
    formatted = "\n".join(f"[{pid}] {summary}" for pid, summary in parts)
    resp = await aclient.messages.create(
        model=SONNET,
        max_tokens=400,
        messages=[{"role": "user", "content":
            "Below are one-sentence summaries of 6 distinct sources. "
            "Fuse them into a single coherent paragraph (≤120 words) that traces "
            "the deep-learning revolution chronologically. Cite source ids inline "
            "as [doc-N] for every claim. Do not invent facts not in the inputs.\n\n"
            + formatted
        }],
    )
    return resp.content[0].text.strip()

items = [(f"doc-{i}", d) for i, d in enumerate(SUMMARIZE_DOCS)]
mr = asyncio.run(map_reduce(
    items, summarize_one, fuse_reducer,
    max_concurrent=4, on_failure=FailurePolicy.MIN_K, min_k=4,
))

print(f"Mapped  : {mr.n_ok}/{mr.n_total} ok in {mr.map_latency_s:.1f}s")
print(f"Reduced : {mr.reduce_latency_s:.1f}s")
print(f"Failed  : {mr.failed_ids}\n")
print("FUSED PARAGRAPH\n" + "─" * 60)
print(mr.final)


## 4 · Self-consistency voting

A celebrated special case of map-reduce: fan the *same* prompt out N times to the *same* model with `temperature>0`, then take the majority answer. Wang et al. (2022, "Self-Consistency Improves Chain of Thought Reasoning") showed this lifts GSM8K accuracy by ~18 points over greedy decoding for the same model.

Why it works: hard problems have many wrong reasoning paths and only a few right ones. Sampling N chains and voting amplifies the right ones.

Why it can fool you: if the model is *biased* (consistently wrong in the same way), N draws don't help — they just give you N copies of the wrong answer at high confidence. Self-consistency only beats greedy when the model is calibrated enough that the right answer is the *plurality* of its sample distribution.

Below we build `self_consistency_vote` and run it on a reasoning problem where the correct answer is 13.

In [ ]:
import re

VOTE_PROMPT = (
    "Q: A bookstore has 47 fiction books and 38 nonfiction. They sell 12 fiction "
    "books and 9 nonfiction, then receive a shipment of 15 fiction and 7 nonfiction. "
    "Two more nonfiction were returned damaged and discarded. How many more fiction "
    "than nonfiction books does the store now have? Think step by step, then end "
    "with the line 'ANSWER: <integer>'."
)

ANSWER_RE = re.compile(r"ANSWER:\s*(-?\d+)", re.IGNORECASE)

async def one_vote(_):
    resp = await aclient.messages.create(
        model=HAIKU, max_tokens=400, temperature=0.9,
        messages=[{"role": "user", "content": VOTE_PROMPT}],
    )
    text = resp.content[0].text
    m = ANSWER_RE.search(text)
    return int(m.group(1)) if m else None

async def self_consistency_vote(n_samples: int = 7) -> dict:
    items = [(f"draw-{i}", i) for i in range(n_samples)]
    results = await parallel_fan_out(items, one_vote, max_concurrent=n_samples)
    answers = [r.value for r in results if r.ok and r.value is not None]
    if not answers:
        return {"answer": None, "tally": {}, "votes": 0}
    tally = Counter(answers)
    top, votes = tally.most_common(1)[0]
    return {"answer": top, "tally": dict(tally), "votes": votes, "n_valid": len(answers)}

verdict = asyncio.run(self_consistency_vote(n_samples=7))
print(f"Self-consistency answer: {verdict['answer']}  "
      f"({verdict['votes']}/{verdict['n_valid']} samples agree)")
print(f"Full tally: {verdict['tally']}")
# Ground truth: 47-12+15=50 fiction; 38-9+7-2=34 nonfiction; 50-34 = 16.
print("(Correct answer: 16)")
# 💡 EXPERIMENT — bump n_samples to 11 and see if the vote sharpens on 16.
# Then drop to n_samples=1 (greedy equivalent) and compare. Self-consistency
# usually beats greedy *only* when the right answer is the model's plurality —
# if Haiku is biased toward a wrong answer, voting locks that wrong answer in
# (cf. failure mode #6 below).


## 5 · Fan-out over A2A (distributed)

The helpers above run inside one Python process — fast to demo, but you do *not* get the isolation properties that motivated A2A in L32 (each agent in its own process, its own deps, its own scaling envelope).

The distributed version is the same shape with `httpx.AsyncClient` in the worker:

```python
async def post_to_agent(client, url, payload, idem_id):
    r = await client.post(f"{url}/tasks/send", json={
        "id": idem_id,
        "message": {"role": "user", "parts": [{"type": "text", "text": payload}]},
    })
    return r.json()["id"]
```

Below we stand up **two** identical "summarizer" A2A agents on different ports (mimicking horizontal scale), then fan 6 summarize-this-doc tasks across them in parallel and reduce.

(In production these would be separate Kubernetes pods. In Colab they're two threaded uvicorn processes — same protocol.)

In [ ]:
# Minimal A2A agent — a stripped-down port of L32 so this notebook is self-contained.
# Each agent runs FastAPI in a daemon thread on its own port.

from fastapi import FastAPI
from pydantic import BaseModel
import threading, uvicorn, httpx

class SendTaskReq(BaseModel):
    id: str | None = None
    message: dict

# Per-port server registry to make this cell idempotent (re-running the cell
# in Colab won't double-bind a port).
_servers: dict[int, threading.Thread] = {}

def make_summarizer_app(agent_name: str) -> FastAPI:
    app = FastAPI()
    tasks: dict[str, dict] = {}  # in-memory store

    @app.get("/.well-known/agent.json")
    def card():
        return {"name": agent_name, "skills": [{"id": "summarize", "name": "Summarize a document"}]}

    @app.post("/tasks/send")
    async def send(req: SendTaskReq):
        tid = req.id or str(uuid.uuid4())
        if tid in tasks and tasks[tid]["state"] in ("completed", "failed"):
            return {"id": tid}  # idempotent — return existing task id

        tasks[tid] = {"state": "working", "result": None}

        async def work():
            try:
                doc = req.message["parts"][0]["text"]
                resp = await aclient.messages.create(
                    model=HAIKU, max_tokens=80,
                    messages=[{"role": "user", "content": f"Summarize in ONE sentence:\n\n{doc}"}],
                )
                tasks[tid] = {"state": "completed", "result": resp.content[0].text.strip()}
            except Exception as e:
                tasks[tid] = {"state": "failed", "result": str(e)}

        asyncio.create_task(work())
        return {"id": tid}

    @app.get("/tasks/{tid}")
    def get_task(tid: str):
        return tasks.get(tid, {"state": "unknown", "result": None}) | {"id": tid}

    return app

def start_agent(port: int, name: str):
    if port in _servers and _servers[port].is_alive():
        print(f"agent on :{port} already running")
        return
    app = make_summarizer_app(name)
    cfg = uvicorn.Config(app, host="127.0.0.1", port=port, log_level="error")
    server = uvicorn.Server(cfg)
    t = threading.Thread(target=server.run, daemon=True)
    t.start()
    _servers[port] = t
    time.sleep(0.5)  # let bind complete
    print(f"started {name} on http://127.0.0.1:{port}")

start_agent(8011, "summarizer-A")
start_agent(8012, "summarizer-B")


In [ ]:
# Parallel fan-out across the two A2A peers. The pool of agent URLs is the
# round-robin target; each task picks the next worker. In production a real
# load balancer (or Kubernetes service) does this — here we do it by hand.

AGENT_POOL = ["http://127.0.0.1:8011", "http://127.0.0.1:8012"]

async def a2a_summarize(doc: str, *, agent_url: str) -> str:
    async with httpx.AsyncClient(timeout=30) as http:
        idem = str(uuid.uuid4())
        r = await http.post(f"{agent_url}/tasks/send",
                            json={"id": idem,
                                  "message": {"parts": [{"type": "text", "text": doc}]}})
        tid = r.json()["id"]

        # Poll until terminal — bounded
        for _ in range(60):
            await asyncio.sleep(0.3)
            r = await http.get(f"{agent_url}/tasks/{tid}")
            body = r.json()
            if body["state"] in ("completed", "failed"):
                if body["state"] == "failed":
                    raise RuntimeError(body.get("result", "agent failure"))
                return body["result"]
        raise TimeoutError(f"task {tid} did not finish on {agent_url}")

async def parallel_a2a_fan_out(docs: list[str], pool: list[str]) -> list[FanoutResult[str]]:
    # Stable id = doc index; round-robin agent assignment
    items = []
    for i, d in enumerate(docs):
        agent = pool[i % len(pool)]
        items.append((f"doc-{i}@{agent.rsplit(':', 1)[1]}",
                      lambda url=agent, doc=d: a2a_summarize(doc, agent_url=url)))

    sem = asyncio.Semaphore(len(pool) * 2)  # 2 in-flight per agent

    async def run_one(item_id, factory):
        async with sem:
            t0 = time.time()
            try:
                v = await factory()
                return FanoutResult(item_id, True, v, None, time.time() - t0)
            except Exception as e:
                return FanoutResult(item_id, False, None, f"{type(e).__name__}: {e}", time.time() - t0)

    return await asyncio.gather(*(run_one(i, f) for i, f in items))

t0 = time.time()
out = asyncio.run(parallel_a2a_fan_out(SUMMARIZE_DOCS, AGENT_POOL))
dt = time.time() - t0

print(f"6 docs across 2 A2A agents in {dt:.1f}s")
for r in out:
    marker = "✓" if r.ok else "✗"
    snippet = (r.value or r.error or "")[:70]
    print(f"{marker} {r.id} ({r.latency_s:.1f}s)  {snippet}")


## 6 · When *not* to fan out — the decision matrix

Map-reduce is the right hammer **only** when items are independent. The moment one item's result influences another's prompt, you're back in coordination territory and the Blackboard (L33) or Debate (L34) shape is correct.

| Pattern | Items relate how? | Best for | Latency | Cost |
|---|---|---|---|---|
| **Map-reduce (L35)** | independent | 20 docs to summarize, 5 self-consistency votes, fact-check N claims | `max(map) + reduce` | `N×` |
| **Blackboard (L33)** | later items read earlier items off shared state | drafting a report where searchers, synthesizer, critic, editor all need to see the current state | serial, accretive | `K×` where K = #ticks |
| **Debate (L34)** | adversarial, strict alternation | adjudicating a defensible question where eval is the bottleneck | strictly serial | `~2 × turns + judge` |
| **A2A 1↔1 (L32)** | linear pipeline | one specialist hands off to another (researcher → critic) | serial | `pipeline length` |

A 3-question decision tree:

1. **Can items be processed in any order without any one looking at any other?** → Map-reduce.
2. **Do producers need to react to each other's running output?** → Blackboard.
3. **Is the *judging* step the bottleneck and you want adversarial pressure?** → Debate.

A common failure I'll save you: people reach for fan-out on "generate 6 sections of a report". The sections need to be coherent with each other — a fan-out gives you 6 essays on related topics that overlap and contradict. Use Blackboard with a Synthesizer KS instead.


## 7 · Failure modes that bite real fan-outs

1. **Cost explosion.** N× is fine for 5 items; check the bill for 500. Cache aggressively (L22 prompt caching), and put a hard `max_concurrent` *and* a hard `max_n_items` cap on the entry point.
2. **`asyncio.gather` cancellation semantics.** Default: first exception cancels every other task in flight and the partial results are lost. **Always** pass `return_exceptions=True` (or use `parallel_fan_out` above which sets it implicitly via try/except).
3. **Poison-pill workers blocking the reduce.** A worker that hangs forever holds a `Semaphore` slot and keeps the reduce starved. Wrap each worker in `asyncio.wait_for(coro, timeout=X)`.
4. **Thundering herd on shared backend.** All N workers slam the same Anthropic key / same DB at t=0. Use `asyncio.Semaphore` (we did) *and* stagger if you have hundreds of items (`await asyncio.sleep(random.uniform(0, jitter))`).
5. **Output-order anchoring.** A reduce step that anchors on whichever map output *arrived first* is non-deterministic. Sort by stable `id` before passing to the reducer (the dict we build does this implicitly when iteration order matches input order — make it explicit).
6. **Self-consistency on a biased model.** If the model is consistently wrong in the same way, voting gives you N copies of the wrong answer at high confidence. Self-consistency *amplifies* the model's existing prior — it doesn't fix calibration. Pair with the L29 ECE harness.
7. **Idempotency miss in distributed fan-out.** If `post_to_agent` retries on a network blip, you'll double-bill *and* get two task ids unless you supply your own `id` on `POST /tasks/send` (the L32 protocol allows this). We did — note the `idem = uuid.uuid4()` line.


## 8 · Homework

1. **`Verifier × 3` quorum.** Take the L26/L27 jailbreak verifier and run it as a self-consistency vote (`n_samples=3`, majority refuse). Measure ΔASR vs single-shot. Expected: ASR drops; FRR may rise.
2. **AutoResearcher draft via self-consistency.** Wire `self_consistency_vote` into the L23/L31 AutoResearcher's draft step (`n_samples=3`, Sonnet, temperature=0.7), then pass the majority draft into the Critic loop. Measure: does composite_safety_score change? Does mean draft latency go up by the expected ~1× (parallel) or 3× (sequential)?
3. **A2A Searcher fan-out.** The L33 Blackboard `Searcher` KS proposes 2–3 snippets per tick. Replace it with `A2ASearcher` that fans 4 parallel A2A calls to a search-specialist agent and reduces (dedupe by URL). Compare wall-clock vs the sequential L33 version.
4. **Failure-policy stress test.** Build a `flaky_summarize_one` that fails 30% of the time (random `RuntimeError`) and run `map_reduce` with each `FailurePolicy` value. Show the per-policy behavior on the same 10-item input.
5. **Self-consistency on a calibrated vs uncalibrated model.** Pick a question your Haiku gets wrong (e.g. a tricky GSM8K item). Run `self_consistency_vote(n=11)`. Then add the L29 `CALIBRATED_SYSTEM` and rerun. Does the vote distribution sharpen toward the right answer?


## 9 · Coming next — L36: Track 2 Capstone

The Track 2 capstone wires every coordination primitive we built in L32–L35 into one **multi-agent research swarm**:

- **A2A** for cross-process specialists (Searcher, Synthesizer, Critic each their own process)
- **Blackboard** for shared state
- **Debate** for "ship or revise" adjudication when the Critic disagrees with the Editor
- **Parallel fan-out** for the Searcher to query N sources concurrently
- All sitting on top of the L24–L31 reliability harness

The capstone is the demo you put in your README. Get this lesson's code into your head — L36 will use these helpers verbatim.
